In [59]:
import numpy as np
import pandas as pd
import warnings

from tqdm import tqdm, TqdmWarning
from datetime import datetime
import light_curve as lc
from pathlib import Path
from typing import Literal, Callable, Any


warnings.filterwarnings("ignore", category=TqdmWarning)

In [60]:
Type = Literal["train", "test"]
Split = Literal["split_01", "split_02", "split_03", "split_04", "split_05", "split_06", "split_07", "split_08", "split_09", "split_10", "split_11", "split_12", "split_13", "split_14", "split_15", "split_16", "split_17", "split_18", "split_19", "split_20"]


EPS = np.finfo(float).eps


def now() -> str:
    return datetime.now().astimezone().strftime("%Y%m%d-%H%M%S-%z")

### Data Loading

In [61]:
def load_log_df(type: Type, **kwargs) -> pd.DataFrame:
    return pd.read_parquet(f"../artifacts/kaggle/{type}_log.parquet", **kwargs)

def load_flc_df(type: Type, split: Split, **kwargs) -> pd.DataFrame:
    return pd.read_parquet(f"../artifacts/kaggle/{split}/{type}_flc.parquet", **kwargs)


def __ingest_dfs(type: Type):
    log_df = pd.read_csv(f"../artifacts/kaggle/{type}_log.csv", index_col="object_id")
    log_df.to_parquet(f"../artifacts/kaggle/{type}_log.parquet")

    splits = sorted(log_df["split"].unique())
    for split in splits:
        flc_df = pd.read_csv(f"../artifacts/kaggle/{split}/{type}_full_lightcurves.csv")
        flc_df.to_parquet(f"../artifacts/kaggle/{split}/{type}_flc.parquet")


__ingest_dfs(type="train")
__ingest_dfs(type="test")

### Feature Engineering

In [67]:
def load_feats_df(type: Type, split: Split, **kwargs):
    return pd.read_parquet(f"../artifacts/feats/{split}/{type}_feats.parquet", **kwargs)


def __build_and_ingest_feats(type: Type):
    log_df = load_log_df(type=type)

    with tqdm(total=len(log_df), unit="obj") as pb:
        for split, log_sub_df in log_df.groupby("split"):
            pb.set_description(f"Building features for `{split}` (`{type}`)")

            feats_buf = []

            for obj_id, log_row in log_sub_df.iterrows():
                flc_df = load_flc_df(type=type, split=split, filters=[("object_id", "==", obj_id)]) # pyright: ignore[reportArgumentType]
                feats = __build_feats_for_obj(log_row, flc_df)
                feats_buf.append(feats)

                pb.update()

            pb.set_description(f"Ingesting features for `{split}` (`{type}`)")

            Path(f"../artifacts/feats/{split}").mkdir(parents=True, exist_ok=True)
            pd.DataFrame(feats_buf).to_parquet(f"../artifacts/feats/{split}/{type}_feats.parquet")


# TODO Integrate `light-curve`/`cesium` for time-series feature engineering
# TODO Add features acquired from domain knowledge
def __build_feats_for_obj(log_row: pd.Series, flc_df: pd.DataFrame) -> dict:
    # Could reorder?
    __de_extinct(log_row, flc_df)

    flc_df["Flux_ratio"] = flc_df["Flux"] / flc_df["Flux_err"]

    feats = log_row.to_dict()

    feats.update(__build_stats_feats_for_obj(flc_df))
    feats.update(__build_lc_feats_for_obj(flc_df))
    feats.update(__build_domain_feats_for_obj(flc_df))

    return feats


def __build_stats_feats_for_obj(flc_df: pd.DataFrame) -> dict:
    feats = {}

    pivot_flc_df = flc_df.pivot_table(index="Time (MJD)", columns="Filter", values=["Flux", "Flux_err", "Flux_ratio"])

    for agg_name, agg in __STATS_AGGS.items():
        for feat in ["Flux", "Flux_err", "Flux_ratio"]:
            feats[f"{feat}_{agg_name}"] = agg(flc_df[feat])
            
            for filter in __FILTERS:
                if filter in pivot_flc_df.columns:
                    feats[f"{feat}_{agg_name}_{filter}"] = agg(pivot_flc_df[(feat, filter)]) # pyright: ignore[reportCallIssue]

    return feats


__STATS_AGGS: dict[str, Callable[[pd.Series], Any]] = {
    "mean": np.mean,
    "std": np.std,
    "min": np.min,
    "max": np.max,
    "median": np.median,
    "q25": lambda feats: feats.quantile(0.25),
    "q75": lambda feats: feats.quantile(0.75),
}
__FILTERS = ["u", "g", "r", "i", "z", "y"]


def __build_lc_feats_for_obj(flc_df: pd.DataFrame) -> dict:   
    feats = {}

    def lc_fe(df: pd.DataFrame):
        return __LC_FE(df.index.to_numpy(dtype=np.float64), df["Flux"].to_numpy(), df["Flux_err"].to_numpy()) # pyright: ignore[reportCallIssue]

    # TODO Try optimizing
    flc_fin_df = flc_df[(np.isfinite(flc_df.index) & np.isfinite(flc_df["Flux"]) & np.isfinite(flc_df["Flux_err"]))]
    
    if len(flc_fin_df) >= __LC_FE_MIN_NROWS:
        lc_feats = lc_fe(flc_fin_df)

        for feat_name, feat in zip(__LC_FE.names, lc_feats): # pyright: ignore[reportAttributeAccessIssue]
            feats[feat_name] = feat

    for filter in __FILTERS:
        flc_fin_sub_df = flc_fin_df.loc[flc_df["Filter"] == filter]

        if len(flc_fin_sub_df) >= __LC_FE_MIN_NROWS:
            lc_feats = lc_fe(flc_fin_sub_df)

            for feat_name, feat in zip(__LC_FE.names, lc_feats): # pyright: ignore[reportAttributeAccessIssue]
                feats[f"{feat_name}_{filter}"] = feat

    return feats


def __build_domain_feats_for_obj(flc_df: pd.DataFrame) -> dict:
    return {}


__LC_FE = lc.Extractor(
    lc.LinearFit(), # pyright: ignore[reportArgumentType]
    lc.StetsonK(), # pyright: ignore[reportArgumentType]
    lc.Amplitude(), # pyright: ignore[reportArgumentType]
    lc.BeyondNStd(), # pyright: ignore[reportArgumentType]
    lc.Skew(), # pyright: ignore[reportArgumentType]
    lc.Kurtosis(), # pyright: ignore[reportArgumentType]
)
__LC_FE_MIN_NROWS = 4


# TODO Consider extinction.fitzpatrick99
def __de_extinct(log_row: pd.Series, flc_sub_df: pd.DataFrame):
    r_λ = flc_sub_df["Filter"].map({
        "u": 4.81,
        "g": 3.64,
        "r": 2.70,
        "i": 2.06,
        "z": 1.58,
        "y": 1.31
    })

    c_λ = np.pow(10, 0.4 * r_λ * log_row["EBV"])

    flc_sub_df["Flux"] *= c_λ
    flc_sub_df["Flux_err"] *= c_λ


__build_and_ingest_feats(type="train")
__build_and_ingest_feats(type="test")

Building features for `split_01` (`train`):   0%|          | 0/3043 [00:00<?, ?obj/s]

Ingesting features for `split_20` (`test`): 100%|██████████| 7135/7135 [01:23<00:00, 85.66obj/s]


### Data Cleaning

In [78]:
train_feats_df = pd.concat([load_feats_df(type="train", split=split) for split in sorted(load_log_df(type="train")["split"].unique())]).drop(columns=["SpecType", "English Translation", "split", "target"])

train_feats_df

,Z,Z_err,EBV,Flux_mean,Flux_err_mean,Flux_ratio_mean,Flux_std,Flux_err_std,Flux_ratio_std,Flux_min,...,skew_z,kurtosis_z,linear_fit_slope_y,linear_fit_slope_sigma_y,linear_fit_reduced_chi2_y,stetson_K_y,amplitude_y,beyond_1_std_y,skew_y,kurtosis_y
0,3.0490,NaN,0.110,1.168616,0.622708,4.149514,5.789107,0.585147,18.428182,-3.147488,...,3.328534,11.318189,0.018636,0.026917,3.581662,0.828991,4.923171,0.090909,1.525677,2.634059
1,0.4324,NaN,0.058,0.429695,0.587684,1.024884,1.488888,0.561601,3.150463,-1.873722,...,3.127142,12.145236,-0.041316,0.005185,5.486354,0.535131,7.036898,0.103448,2.983446,11.688474
2,0.4673,NaN,0.577,5.218302,1.305302,6.251681,6.401231,0.998163,6.124101,-12.840542,...,0.046864,-1.455541,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.6946,NaN,0.012,0.385203,0.428335,1.657185,0.875471,0.472227,2.612474,-7.753266,...,0.519980,0.730480,0.000576,0.000398,1.351590,0.798893,6.592584,0.226087,-0.967255,3.308255
4,0.4161,NaN,0.058,0.260234,0.482768,0.888452,1.276400,0.461308,4.496637,-3.282238,...,-0.159916,0.382455,-0.018741,0.008021,1.603544,0.814502,2.912294,0.428571,-0.802999,-0.558952
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148,0.8898,NaN,0.042,0.492624,0.555184,1.359930,1.062509,0.501120,2.312394,-1.493477,...,0.679493,0.152494,0.017091,0.006070,1.234541,0.863899,4.128050,0.250000,1.147746,1.782013
149,0.9598,NaN,0.042,0.422353,0.523150,1.416870,1.614420,0.501490,5.103314,-5.607301,...,3.484227,14.988520,-0.013627,0.007295,1.095413,0.825562,4.086785,0.166667,-1.715446,4.783379
150,0.1543,NaN,0.024,0.392762,0.518548,1.253894,1.110451,0.540616,2.772301,-2.854501,...,1.157686,0.265259,-0.015444,0.004689,1.915739,0.751998,4.044818,0.290323,1.061887,1.467398
151,1.1520,NaN,0.019,0.368474,0.572626,1.136057,1.119359,0.633723,5.032753,-2.962387,...,-0.403714,1.404314,-0.002501,0.005412,1.202865,0.785969,2.920671,0.296296,-0.412847,-0.091330


### Training

### Inference

In [79]:
# __dirpath = Path("../artifacts/predictions")
# __dirpath.mkdir(parents=True, exist_ok=True)

# prediction_df.to_csv(f"{__dirpath}/submission-{now()}.csv", index=False)

In [80]:
# TODO
# Assuming GB, no imputation, no scaling, only minimal cleaning.